# Stage 4: Neural Try-On Synthesis

**Goal:** Replace the flat, geometric overlay from Stage 3 with a photorealistic result —  
one that has correct fabric wrinkles, shadows, lighting, and seamless edges.

**Why neural synthesis?**  
Geometric warping (Stage 3) correctly *places* the garment, but it cannot:  
- Add fabric folds and wrinkles that match the body shape  
- Adjust lighting/shadows to match the scene  
- Blend edges seamlessly with skin/hair  

A neural network trained on thousands of (person, garment, result) triplets *learns*  
what real clothing looks like on real bodies, and can hallucinate these details convincingly.

**Approach: Paint-by-Example (Reference-based Inpainting)**  
We use `Fantasy-Studio/Paint-by-Example` — a diffusion model that takes:
- **Input image:** the agnostic person (shirt erased) from Stage 2  
- **Mask:** the erase mask from Stage 2 (region to fill)  
- **Reference image:** the clothing product photo  

The model fills the masked region so it looks like the person is wearing the reference garment.

**How it connects to our pipeline:**
```
Stage 1: Pose          Stage 2: Segmentation      Stage 3: Warping
(body landmarks)  →   (agnostic + erase mask)  →  (warped clothing)
                              |
                              v
                    Stage 4: Neural Synthesis  ←  Clothing Image
                    (Paint-by-Example fills
                     the masked region with
                     photorealistic clothing)
                              |
                              v
                     Photorealistic Try-On
```

**Note on production models:**  
Paint-by-Example is a general reference inpainting model. Production try-on systems  
(IDM-VTON, CatVTON, HR-VITON) are specifically fine-tuned on clothing datasets and  
give better garment fidelity. We use Paint-by-Example here because it requires  
no repo cloning, runs on T4, and clearly demonstrates the concept.

## Cell 1: Install Dependencies

In [ ]:
!pip install diffusers==0.21.4 transformers accelerate -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 2: Mount Drive + Load All Previous Outputs

We load everything built in Stages 1-3:
- `agnostic_person.png` — person with shirt erased (Stage 2)
- `erase_mask.png` — pixel mask of the shirt region (Stage 2)
- `clothing.jpg` — the flat product clothing image (Stage 3)
- `test_person.jpg` — original person photo (for final comparison)

In [ ]:
from google.colab import drive
import os, cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

drive.mount('/content/drive')

BASE_DIR   = '/content/drive/MyDrive/VirtualTryOn'
INPUT_DIR  = os.path.join(BASE_DIR, 'input_images')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output_images')

# Load all Stage 1-3 outputs
person_img   = Image.open(os.path.join(INPUT_DIR,  'test_person.jpg')).convert('RGB')
agnostic_img = Image.open(os.path.join(OUTPUT_DIR, 'agnostic_person.png')).convert('RGB')
erase_mask   = Image.open(os.path.join(OUTPUT_DIR, 'erase_mask.png')).convert('L')
clothing_img = Image.open(os.path.join(INPUT_DIR,  'clothing.jpg')).convert('RGB')
stage3_blend = Image.open(os.path.join(OUTPUT_DIR, 'stage3_blended.png')).convert('RGB')

print(f"Person:    {person_img.size}")
print(f"Agnostic:  {agnostic_img.size}")
print(f"Mask:      {erase_mask.size}")
print(f"Clothing:  {clothing_img.size}")
print(f"Stage3:    {stage3_blend.size}")

# Quick preview of inputs
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, img, title in zip(axes,
    [person_img, agnostic_img, erase_mask, clothing_img],
    ['Original Person', 'Agnostic (Stage 2)', 'Erase Mask (Stage 2)', 'Clothing']):
    ax.imshow(img, cmap='gray' if title == 'Erase Mask (Stage 2)' else None)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Cell 3: Prepare Inputs for Neural Try-On

Paint-by-Example requires all inputs at **512×512** resolution.  
We need to:
1. Resize person + agnostic + mask to 512×512 (preserving content with padding if needed)
2. Resize clothing reference to 512×512
3. Dilate the mask slightly so the model has room to blend edges

**Why mask dilation?**  
The erase mask from Stage 2 is tight around the shirt pixels. If the mask edge falls  
exactly on the shirt boundary, the model has no 'overlap zone' to blend the new garment  
with the surrounding skin/background. Dilating by ~10px gives it that blending room.

In [ ]:
TARGET_SIZE = (512, 512)

def resize_pad(img, size):
    """Resize keeping aspect ratio, pad to square."""
    img.thumbnail(size, Image.LANCZOS)
    padded = Image.new(img.mode, size, (128, 128, 128) if img.mode == 'RGB' else 0)
    offset = ((size[0] - img.size[0]) // 2, (size[1] - img.size[1]) // 2)
    padded.paste(img, offset)
    return padded, offset, img.size

# Resize all inputs
agnostic_512, pad_offset, orig_size = resize_pad(agnostic_img.copy(), TARGET_SIZE)
person_512,   _,          _         = resize_pad(person_img.copy(),   TARGET_SIZE)
clothing_512, _,          _         = resize_pad(clothing_img.copy(), TARGET_SIZE)

# Resize mask and dilate
mask_512, _, _ = resize_pad(erase_mask.copy(), TARGET_SIZE)
mask_np = np.array(mask_512)
kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
mask_dilated = cv2.dilate(mask_np, kernel, iterations=2)
mask_512_dilated = Image.fromarray(mask_dilated)

print(f"All inputs resized to: {TARGET_SIZE}")
print(f"Pad offset applied: {pad_offset}  |  Original size: {orig_size}")

# Preview
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, img, title in zip(axes,
    [person_512, agnostic_512, mask_512_dilated, clothing_512],
    ['Person 512×512', 'Agnostic 512×512', 'Mask (dilated)', 'Clothing 512×512']):
    ax.imshow(img, cmap='gray' if 'Mask' in title else None)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Cell 4: Load the Paint-by-Example Pipeline

**How Paint-by-Example works:**  
It's a Stable Diffusion inpainting model with a special image encoder (CLIP-based) that  
encodes the reference image (clothing) into an embedding. This embedding then guides  
the diffusion denoising process in the masked region, steering it to generate pixels  
that look like the reference image applied to the masked area.

Architecture:  
```
Agnostic Image + Mask → [Noised Latent]
Clothing Image        → [CLIP Image Encoder] → reference embedding
                                                      |
                        [U-Net Denoiser] ← cross-attention
                                |
                         [VAE Decoder]
                                |
                        Photorealistic Output
```

The model downloads ~5GB on first run — it caches automatically.

In [ ]:
import torch
from diffusers import PaintByExamplePipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on: {device}")

pipe = PaintByExamplePipeline.from_pretrained(
    "Fantasy-Studio/Paint-by-Example",
    torch_dtype=torch.float16
).to(device)

# Memory optimization for T4
pipe.enable_attention_slicing()

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Cell 5: Run Neural Try-On Inference

**Key parameters:**
- `num_inference_steps`: more steps = better quality but slower (50 is a good balance)
- `guidance_scale`: how closely to follow the reference image (5-10 is typical)
- `generator`: random seed — change this to get different variations

We run 3 variations with different seeds and pick the best one.  
This is standard practice in diffusion-based generation — the model is stochastic,  
so multiple seeds give slightly different results in fabric texture and wrinkle placement.

In [ ]:
import torch

SEEDS          = [42, 123, 777]
NUM_STEPS      = 50
GUIDANCE_SCALE = 7.5

results = []

for i, seed in enumerate(SEEDS):
    print(f"Running inference — seed {seed} ({i+1}/{len(SEEDS)})...", end=' ')
    generator = torch.Generator(device=device).manual_seed(seed)

    output = pipe(
        image          = agnostic_512,
        mask_image     = mask_512_dilated,
        example_image  = clothing_512,
        num_inference_steps = NUM_STEPS,
        guidance_scale = GUIDANCE_SCALE,
        generator      = generator,
    )

    result = output.images[0]
    results.append(result)

    # Save each variation
    save_path = os.path.join(OUTPUT_DIR, f'neural_tryon_seed{seed}.png')
    result.save(save_path)
    print(f"saved.")

print(f"\nAll {len(SEEDS)} variations saved to Drive.")

# Display all variations
fig, axes = plt.subplots(1, len(SEEDS), figsize=(16, 6))
for ax, result, seed in zip(axes, results, SEEDS):
    ax.imshow(result)
    ax.set_title(f'Seed {seed}')
    ax.axis('off')
plt.suptitle('Neural Try-On Variations', fontsize=13)
plt.tight_layout()
plt.show()

## Cell 6: Post-Process — Restore Face and Paste Back onto Original

The neural output is 512×512. We need to:
1. **Restore the face/hair** — paste original face pixels back (diffusion sometimes drifts on faces)
2. **Paste back onto the original-resolution image** — undo the 512×512 padding

This gives us the final full-resolution try-on result.

In [ ]:
# Pick the best-looking result (change index if another seed looks better)
BEST_IDX = 0
best_result = results[BEST_IDX]

# --- Restore face and hair from agnostic image ---
# The model should not have changed these, but we enforce it for safety
seg_map  = np.load(os.path.join(OUTPUT_DIR, 'seg_map.npy'))

# Resize seg_map to 512x512 for face restoration
seg_512  = cv2.resize(seg_map.astype(np.uint8), TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
face_mask_512 = ((seg_512 == 11) | (seg_512 == 2)).astype(np.uint8) * 255  # face + hair

# Dilate slightly to avoid hard edges around the face
fk = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
face_mask_512 = cv2.dilate(face_mask_512, fk, iterations=1)
face_alpha = face_mask_512.astype(np.float32) / 255.0
face_3ch   = np.stack([face_alpha] * 3, axis=-1)

result_np   = np.array(best_result).astype(np.float32)
agnostic_np = np.array(agnostic_512).astype(np.float32)

restored = (agnostic_np * face_3ch + result_np * (1.0 - face_3ch)).astype(np.uint8)
final_512 = Image.fromarray(restored)

# --- Paste back onto original resolution ---
# Crop out the padded region to get back to original aspect ratio
ox, oy   = pad_offset
ow, oh   = orig_size

# Scale factor from original → 512
scale = min(TARGET_SIZE[0] / person_img.size[0], TARGET_SIZE[1] / person_img.size[1])
scaled_w = int(person_img.size[0] * scale)
scaled_h = int(person_img.size[1] * scale)

# Crop the padded area out
cropped = final_512.crop((ox, oy, ox + scaled_w, oy + scaled_h))

# Resize back to original resolution
final_full = cropped.resize(person_img.size, Image.LANCZOS)

# Save final result
final_path = os.path.join(OUTPUT_DIR, 'final_tryon.png')
final_full.save(final_path)
print(f"Final try-on result saved to: {final_path}")
print(f"Output size: {final_full.size}")

## Cell 7: Final Pipeline Comparison

The complete journey — from raw inputs to neural try-on output.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(26, 8))

panels = [
    (person_img,    'Original Person'),
    (clothing_img,  'Clothing Item'),
    (agnostic_img,  'Cloth-Agnostic\n(Stage 2)'),
    (stage3_blend,  'Geometric Warp\n(Stage 3)'),
    (final_full,    'Neural Try-On\n(Stage 4)'),
]

for ax, (img, title) in zip(axes, panels):
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Virtual Try-On — Full Pipeline', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'full_pipeline.jpg'), dpi=150, bbox_inches='tight')
plt.show()

print("\nAll outputs saved to Drive:")
for fname in ['neural_tryon_seed42.png', 'neural_tryon_seed123.png',
              'neural_tryon_seed777.png', 'final_tryon.png', 'full_pipeline.jpg']:
    p = os.path.join(OUTPUT_DIR, fname)
    print(f"  {'OK' if os.path.exists(p) else 'MISSING'} {fname}")

---
## What we learned in Stage 4

- **Reference-based inpainting** uses a clothing image as a visual prompt to fill the masked torso region
- The diffusion model handles wrinkles, shadows, and edge blending that geometric methods cannot
- Multiple seeds produce variations — pick the best one
- Face/hair restoration ensures the model doesn't hallucinate on areas it shouldn't touch

### How to get even better results (next steps):

| Improvement | How |
|---|---|
| Better clothing fidelity | Use IDM-VTON or CatVTON (fine-tuned on fashion data) |
| Full-body (pants/dresses) | Add DressCode dataset, use lower-body segmentation labels |
| Real-time inference | Quantize model to INT8, use TensorRT |
| Production API | Wrap pipeline in FastAPI, deploy on RunPod/Modal |

### Complete pipeline summary:
```
Person Photo + Clothing Photo
         |
         v
[Stage 1] Pose Estimation      → body skeleton (33 landmarks)
[Stage 2] Body Segmentation    → cloth-agnostic person + erase mask
[Stage 3] Clothing Warping     → geometrically placed garment
[Stage 4] Neural Synthesis     → photorealistic try-on result
         |
         v
   Final Try-On Image
```